# 04 - Feature Engineering

Create features for demand forecasting.

## Objectives:
- Create time-based features
- Create lag features
- Create rolling statistics
- Create price features
- Create event features
- Save engineered dataset

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np

from src.feature_engineering import FeatureEngineer
from src.utils import load_dataframe, save_dataframe

import warnings
warnings.filterwarnings('ignore')

## 1. Load Cleaned Data

In [ ]:
df = load_dataframe('../data/processed/cleaned_data.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"Data loaded: {df.shape}")
print(f"\nColumns before feature engineering:")
print(list(df.columns))

## 2. Sample Data (Optional)

For faster processing, work with a sample. Comment out for full dataset.

In [ ]:
# Uncomment to work with a sample
# df = df.sample(frac=0.1, random_state=42)
# print(f"Working with sample: {df.shape}")

## 3. Create All Features

In [ ]:
engineer = FeatureEngineer()

print("Creating all features...")
df_engineered = engineer.create_all_features(df)

print(f"\nFeature engineering complete!")
print(f"Shape: {df_engineered.shape}")
print(f"New features created: {df_engineered.shape[1] - df.shape[1]}")

## 4. Review Created Features

In [ ]:
new_features = [col for col in df_engineered.columns if col not in df.columns]

print(f"New features created ({len(new_features)}):")
for i, feature in enumerate(new_features, 1):
    print(f"{i:2d}. {feature}")

In [ ]:
print("\nSample of engineered features:")
df_engineered[new_features[:10]].head()

## 5. Check Missing Values

In [ ]:
missing_counts = df_engineered.isnull().sum()
missing_features = missing_counts[missing_counts > 0].sort_values(ascending=False)

if len(missing_features) > 0:
    print("Features with missing values:")
    print(missing_features)
    print(f"\nTotal missing: {missing_counts.sum()}")
else:
    print("No missing values!")

## 6. Remove Initial Rows with NaN

Lag and rolling features create NaN for initial rows

In [ ]:
print(f"Rows before removing NaN: {len(df_engineered)}")

df_final = df_engineered.dropna()

print(f"Rows after removing NaN: {len(df_final)}")
print(f"Rows removed: {len(df_engineered) - len(df_final)}")

## 7. Feature Correlation with Target

In [ ]:
numeric_features = df_final.select_dtypes(include=[np.number]).columns
correlations = df_final[numeric_features].corr()['sales'].sort_values(ascending=False)

print("Top 20 features correlated with sales:")
print(correlations.head(20))

## 8. Save Engineered Dataset

In [ ]:
save_dataframe(df_final, '../data/processed/engineered_features.csv')

print("\nEngineered dataset saved!")
print(f"Final shape: {df_final.shape}")
print(f"Date range: {df_final['date'].min()} to {df_final['date'].max()}")
print(f"Total features: {df_final.shape[1]}")

## Summary

Feature engineering completed:
- Time-based features (year, month, day, etc.)
- Lag features (1, 7, 14, 28 days)
- Rolling statistics (mean, std, min, max)
- Price features (changes, momentum)
- Event features
- SNAP features
- Aggregated features

Next: Model Training